# 07 — Candidate Co-mention Network

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("07_candidate_comention_network", total_steps=6, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook builds the candidate co-mention network. Candidates are nodes; an edge exists when two candidates are mentioned in the same tweet. Co-mention means discourse association, not necessarily support.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
tweets = pd.read_csv(PROCESSED / "tweets_with_entities.csv", parse_dates=["createdAt"])
tweets = parse_list_columns(tweets, ["candidates_mentioned"])
edges = make_pair_edges(tweets["candidates_mentioned"], source_col="source", target_col="target")
edges.to_csv(PROCESSED / "candidate_comention_edges.csv", index=False)
print("Candidate co-mention edges:", len(edges))
edges.head(20)


In [ ]:
progress.step("Purpose")
G = graph_from_edges(edges, source="source", target="target", weight="weight", directed=False)
summary = network_summary(G, "candidate_comention")
metrics = compute_network_metrics(G)
communities = detect_communities(G)
metrics["community"] = metrics["node"].map(communities)
summary.to_csv(TABLES / "07_candidate_network_summary.csv", index=False)
metrics.to_csv(PROCESSED / "candidate_comention_node_metrics.csv", index=False)
nx.write_gexf(G, NETWORKS / "candidate_comention_network.gexf")
display(summary)
display(metrics.head(20))


In [ ]:
progress.step("Purpose")
fig = px.bar(metrics.sort_values("pagerank", ascending=True), x="pagerank", y="node", orientation="h", title="Candidates by PageRank in the co-mention network")
fig.update_layout(height=900, yaxis_title="Candidate")
save_plotly(fig, INTERACTIVE / "07_candidate_pagerank.html", FIGURES / "07_candidate_pagerank.png")
fig.show()


In [ ]:
progress.step("Purpose")
# Candidate pair heatmap for strongest co-mentions
if not edges.empty:
    top_nodes = metrics.sort_values("weighted_degree", ascending=False).head(25)["node"].tolist()
    matrix_edges = edges[edges["source"].isin(top_nodes) & edges["target"].isin(top_nodes)]
    pivot = pd.DataFrame(0, index=top_nodes, columns=top_nodes, dtype=float)
    for _, r in matrix_edges.iterrows():
        pivot.loc[r["source"], r["target"]] = r["weight"]
        pivot.loc[r["target"], r["source"]] = r["weight"]
    plt.figure(figsize=(16, 14))
    sns.heatmap(pivot, cmap="rocket_r", linewidths=.5)
    plt.title("Candidate co-mention heatmap: top 25 by weighted degree")
    plt.tight_layout()
    plt.savefig(FIGURES / "07_candidate_comention_heatmap.png", dpi=220, bbox_inches="tight")
    plt.show()


In [ ]:
progress.step("save_pyvis_network(G, INTERACTIVE / '07_candidate_comention_network_interactive.html', title='Candidate co-mention netwo")
save_pyvis_network(G, INTERACTIVE / "07_candidate_comention_network_interactive.html", title="Candidate co-mention network", node_metrics=metrics, max_nodes=100)
print("Saved:", INTERACTIVE / "07_candidate_comention_network_interactive.html")


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
